# OpenAI Agents SDK — From Primitives to Multi-Agent Handoffs
## Primitives, Guardrails, Handoffs — The OpenAI Agents Stack
⏱ ~15 min

The **OpenAI Agents SDK** (released early 2025) is the official Python library for building production-grade agentic workflows directly on top of the OpenAI platform. Unlike orchestration frameworks that wrap the API (LangGraph, AutoGen), the Agents SDK bakes routing, tool calling, tracing, and agent-to-agent handoffs into first-class primitives — with zero boilerplate.

By the end of this session you will understand *every primitive*, know *when* to use handoffs vs. graph edges, and have a working multi-agent triage system you built yourself.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — SDK architecture and core primitives |
| 2 | **First Agent** — `Agent` + `Runner.run_sync()` minimal example |
| 3 | **Tools** — `@tool` decorator, docstrings as schemas |
| 4 | **Handoffs** — Agent-to-agent routing with `handoff()` |
| 5 | **Multi-Agent System** — Triage → Researcher + Writer |
| 6 | **Tracing** — Inspecting `RunResult` and built-in step recording |
| 7 | **Guardrails** — Input validation patterns with structured output |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with the project `.venv` activated  **or**  Google Colab (install cell below handles it)
- An `OPENAI_API_KEY` in `.env` (local) or Colab Secrets

### Key References
> OpenAI Agents SDK documentation: https://openai.github.io/openai-agents-python/
> Yao, S., Zhao, J., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629
> Xi, Z., Chen, W., et al. (2023). *The Rise and Potential of Large Language Model Based Agents: A Survey.* https://arxiv.org/abs/2309.07864
> OpenAI function calling reference: https://platform.openai.com/docs/guides/function-calling

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "openai-agents",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ──────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon) as OPENAI_API_KEY
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ─────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or invalid. "
    "Local: add to .env  |  Colab: add to Secrets panel"
)
print(f"OPENAI_API_KEY present: True  (starts with sk-{key[3:6]}...)")

---

## Part 1 — Concepts: SDK Architecture and Core Primitives

---

### Why Another Agent Framework?

Before 2025, building a multi-agent system required choosing between:
- **LangGraph** — explicit state machines with typed `TypedDict` state and graph edges
- **AutoGen** — conversational round-robin agents with `GroupChat`
- **CrewAI** — role-based crews with sequential or hierarchical processes

The OpenAI Agents SDK takes a different bet: agents are simple Python objects with *declarative routing* via `handoffs`. No graph, no state machine, no boilerplate — the SDK and the model handle routing implicitly through conversation history.

---

### SDK Architecture

```
  ┌─────────────────────────────────────────────────────────┐
  │                   OpenAI Agents SDK                     │
  │                                                         │
  │   ┌──────────┐    ┌──────────┐    ┌──────────────────┐ │
  │   │  Agent   │───▶│  Runner  │───▶│    RunResult     │ │
  │   │          │    │          │    │  .final_output   │ │
  │   │ name     │    │ run_sync │    │  .last_agent     │ │
  │   │ instruct │    │ run      │    │  .new_items      │ │
  │   │ model    │    │ run_str  │    └──────────────────┘ │
  │   │ tools ───┼──┐ └──────────┘                         │
  │   │ handoffs ┼┐ │                                       │
  │   └──────────┘│ │  ┌──────────────────────────────┐    │
  │               │ └─▶│  @tool  (Python function)    │    │
  │               │    │  docstring → JSON schema      │    │
  │               │    │  return value → tool output   │    │
  │               │    └──────────────────────────────┘    │
  │               │                                         │
  │               │    ┌──────────────────────────────┐    │
  │               └───▶│  handoff(agent)              │    │
  │                    │  transfers control to another │    │
  │                    │  Agent mid-conversation        │    │
  │                    └──────────────────────────────┘    │
  │                                                         │
  │   Built-in tracing: every step recorded automatically   │
  └─────────────────────────────────────────────────────────┘
```

---

### The Four Core Primitives

| Primitive | What it is | Analogy |
|-----------|-----------|--------|
| `Agent` | An LLM persona with instructions, tools, and optional handoffs | A team member with a job description |
| `@tool` | A Python function the model can call during reasoning | A tool in the team member's toolkit |
| `handoff(agent)` | Transfers control from one agent to another | Escalating a ticket to a specialist |
| `Runner` | Executes an agent (sync or async) and returns a `RunResult` | The runtime that manages the conversation loop |

---

### OpenAI Agents SDK vs LangGraph vs AutoGen

| Concern | OpenAI Agents SDK | LangGraph | AutoGen |
|---------|-------------------|-----------|--------|
| **Routing** | `handoff(agent)` — implicit via model | `add_conditional_edges` — explicit graph | `GroupChat` — round-robin or LLM-selected |
| **State** | Conversation history (implicit) | `TypedDict` (explicit, typed) | Message list per agent |
| **Tools** | `@tool` decorator | `@tool` (LangChain) or node function | Function registered on agent |
| **Tracing** | Built-in, zero config | Requires LangSmith or custom callback | Requires external setup |
| **Execution** | `Runner.run_sync()` | `graph.compile().invoke()` | `initiate_chat()` |
| **Best for** | Chat-style handoffs, OpenAI-native stack | Complex state machines, DAGs | Multi-agent debates, round-robin |

> **When to choose Agents SDK:** Your routing logic is expressible in natural language instructions, you want zero-config tracing, and you're building on OpenAI models. Use LangGraph when you need explicit typed state, complex conditional branching, or non-OpenAI providers.

---

## Part 2 — Your First Agent: `Agent` + `Runner.run_sync()`

---

### The `Agent` Object

```python
from agents import Agent

agent = Agent(
    name="MyAgent",          # display name (used in tracing)
    instructions="...",      # system prompt — the agent's job description
    model="gpt-5-nano",      # any OpenAI model
    tools=[],                # list of @tool functions (optional)
    handoffs=[],             # list of handoff(other_agent) (optional)
)
```

### The `Runner`

| Method | Signature | When to use |
|--------|-----------|------------|
| `Runner.run_sync()` | `(agent, input_str) -> RunResult` | Notebooks, scripts, synchronous code |
| `Runner.run()` | `async (agent, input_str) -> RunResult` | FastAPI, async web servers |
| `Runner.run_streamed()` | `async (agent, input_str) -> AsyncIterator` | Streaming UI responses |

### The `RunResult`

| Attribute | Type | What it contains |
|-----------|------|------------------|
| `.final_output` | `str` | The last text response from any agent in the run |
| `.last_agent` | `Agent` | Which agent produced the final output |
| `.new_items` | `list` | All messages and tool calls that occurred during the run |

In [ ]:
# ── 2-A: Minimal Agent — hello world ─────────────────────────────────────────
# The simplest possible agent: a name, instructions, and a model.
# No tools, no handoffs — just an LLM with a system prompt.

from agents import Agent, Runner

MODEL = "gpt-5-nano"  # fast, cheap — good for teaching

# instructions becomes the system prompt — the model's fixed persona for the entire run
hello_agent = Agent(
    name="GreetingAgent",
    instructions=(
        "You are a friendly assistant. "
        "Always respond in exactly one sentence."
    ),
    model=MODEL,
)

# Runner.run_sync() is the synchronous entry point; use await Runner.run() in async contexts
result = Runner.run_sync(hello_agent, "What is the OpenAI Agents SDK?")
print(f"Answer: {result.final_output}")
print(f"Answered by: {result.last_agent.name}")

In [ ]:
# ── 2-B: Inspect the RunResult in detail ─────────────────────────────────────
# new_items contains every message and tool call produced during the run.
# In a single-agent, no-tool run this is just one message item.

result2 = Runner.run_sync(hello_agent, "Name three benefits of agent-based AI systems.")

print(f"final_output: {result2.final_output}")
print(f"last_agent:   {result2.last_agent.name}")
print(f"new_items:    {len(result2.new_items)} item(s) in this run")
print()

for i, item in enumerate(result2.new_items):
    item_type = type(item).__name__
    print(f"  Item {i}: {item_type}")

In [ ]:
# ── 2-C: Instructions matter — same model, different persona ──────────────────
# The instructions field is the system prompt. Different instructions produce
# completely different agent behaviors on identical questions.

concise_agent = Agent(
    name="ConciseAgent",
    instructions="Answer every question in exactly one bullet point. Be terse.",
    model=MODEL,
)

verbose_agent = Agent(
    name="VerboseAgent",
    instructions=(
        "You are a thorough technical writer. "
        "Always provide a detailed multi-paragraph explanation with examples."
    ),
    model=MODEL,
)

question = "What is a function call in the OpenAI API?"

for agent in [concise_agent, verbose_agent]:
    r = Runner.run_sync(agent, question)
    print(f"--- {agent.name} ---")
    print(r.final_output[:300])
    print()

---

## Part 3 — Tools: Giving Agents Capabilities

---

### How `@tool` Works

The `@tool` decorator converts any Python function into a tool the model can call during reasoning. The SDK inspects the function signature and docstring to automatically build the JSON schema that OpenAI's function-calling API requires.

```python
from agents import tool

@tool
def my_tool(query: str) -> str:
    """One sentence description — this becomes the tool description for the model."""
    return f"Result for: {query}"
```

**Rules for good tools:**

| Rule | Why |
|------|----|
| Type-annotate every parameter | The SDK uses annotations to build the JSON schema |
| Write a clear one-sentence docstring | This is what the model reads to decide *when* to call the tool |
| Return a plain string | The model receives the return value as a string in the conversation |
| Keep side effects minimal | Tools can call APIs, read files, query DBs — but be explicit in the docstring |

### Tool Call Flow

```
User question → Agent (LLM) decides to call a tool
                      │
                      ▼
               @tool function executes in Python
                      │
                      ▼
               Return value injected into conversation
                      │
                      ▼
               Agent generates final answer using tool output
```

In [ ]:
# ── 3-A: Define a @tool and attach it to an agent ────────────────────────────
# keyword_search is a simple in-memory retrieval tool.
# The model calls it automatically when it needs facts about the SDK.

from agents import tool

KNOWLEDGE_BASE = [
    "OpenAI Agents SDK was released by OpenAI in early 2025.",
    "OpenAI Agents SDK uses Agent, Runner, tool, and handoff as its core primitives.",
    "An Agent is defined with name, instructions, model, tools, and optional handoffs.",
    "Handoffs transfer control from one agent to another, like routing in LangGraph.",
    "Runner.run_sync() executes an agent synchronously and returns a RunResult.",
    "Built-in tracing in the Agents SDK records every step without external setup.",
    "The @tool decorator wraps a Python function into a tool callable by any Agent.",
    "LangGraph uses nodes and edges to build graphs; Agents SDK uses handoffs.",
    "The Agents SDK supports parallel tool calls and structured output natively.",
    "Agents SDK agents can share tools — a tool defined once can be used by any agent.",
]


# @tool wraps the function in a schema the model can call — the docstring becomes the tool description
@tool
# type hints are required: they define the JSON schema fields the model must supply
def keyword_search(query: str) -> str:
    """Search the knowledge base for facts about the OpenAI Agents SDK."""
    words = set(query.lower().split())
    scored = [(sum(w in doc.lower() for w in words), doc) for doc in KNOWLEDGE_BASE]
    scored.sort(reverse=True)
    top = [doc for score, doc in scored[:3] if score > 0]
    return "\n".join(top) if top else "No relevant facts found."


# Attach the tool to an agent
researcher = Agent(
    name="Researcher",
    model=MODEL,
    instructions=(
        "You answer factual questions about the OpenAI Agents SDK. "
        "Always call keyword_search first to retrieve relevant facts, "
        "then compose a clear 2-3 sentence answer."
    ),
    tools=[keyword_search],
)

result = Runner.run_sync(researcher, "What are the core primitives of the OpenAI Agents SDK?")
print(f"Answer: {result.final_output}")
print(f"Steps:  {len(result.new_items)} items in run")

In [ ]:
# ── 3-B: Multi-parameter tools ────────────────────────────────────────────────
# Tools can accept multiple parameters of different types.
# The model passes values inferred from the user's question.


@tool
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression and return the numeric result."""
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return "Error: only arithmetic expressions are allowed."
        return str(eval(expression))  # noqa: S307  # safe: restricted charset
    except Exception as e:
        return f"Calculation error: {e}"


@tool
def word_count(text: str) -> str:
    """Count the number of words in a text string."""
    count = len(text.split())
    return f"{count} words"


multi_tool_agent = Agent(
    name="MultiToolAgent",
    model=MODEL,
    instructions=(
        "You are a helpful assistant. "
        "Use calculate for math questions and word_count for counting words."
    ),
    tools=[calculate, word_count],
)

questions = [
    "What is (123 * 456) + 789?",
    "How many words are in: 'The quick brown fox jumps over the lazy dog'?",
]

for q in questions:
    r = Runner.run_sync(multi_tool_agent, q)
    print(f"Q: {q}")
    print(f"A: {r.final_output}")
    print()

In [ ]:
# ── 3-C: Shared tools — one tool, multiple agents ─────────────────────────────
# A @tool function is just a Python object. You can attach it to any number of
# agents without redefining it — DRY principle for agent systems.

agent_a = Agent(
    name="AgentA",
    model=MODEL,
    instructions="You answer questions using keyword_search. Be brief.",
    tools=[keyword_search],  # shared tool
)

agent_b = Agent(
    name="AgentB",
    model=MODEL,
    instructions=(
        "You explain technical concepts clearly. "
        "Use keyword_search to ground your explanation in facts."
    ),
    tools=[keyword_search],  # same shared tool
)

q = "What is built-in tracing in the Agents SDK?"

for agent in [agent_a, agent_b]:
    r = Runner.run_sync(agent, q)
    print(f"--- {agent.name} ---")
    print(r.final_output[:200])
    print()

print("Both agents called the same keyword_search tool — no duplication.")

---

## Part 4 — Handoffs: Agent-to-Agent Routing

---

### What Is a Handoff?

A **handoff** is the Agents SDK primitive for transferring control between agents. When Agent A includes `handoff(agent_b)` in its `handoffs` list, the model can decide mid-conversation to pass the user's request to Agent B. Agent B receives the full conversation history — it has complete context.

```python
from agents import Agent, handoff

specialist = Agent(name="Specialist", instructions="...", model=MODEL)

router = Agent(
    name="Router",
    instructions="If the user asks X, hand off to Specialist.",
    model=MODEL,
    handoffs=[handoff(specialist)],  # router can transfer to specialist
)
```

### Handoffs vs. LangGraph Edges

```
LangGraph approach (explicit graph):
─────────────────────────────────────────────────────────────────

  graph = StateGraph(State)
  graph.add_node("triage", triage_fn)
  graph.add_node("researcher", researcher_fn)
  graph.add_node("writer", writer_fn)
  graph.add_conditional_edges(
      "triage",
      route,
      {"researcher": "researcher", "writer": "writer"}
  )
  compiled = graph.compile()


Agents SDK approach (implicit handoffs):
─────────────────────────────────────────────────────────────────

  researcher = Agent(name="Researcher", ...)
  writer     = Agent(name="Writer", ...)
  triage     = Agent(
      name="Triage",
      instructions="Route factual → Researcher, writing → Writer.",
      handoffs=[handoff(researcher), handoff(writer)],
  )
  Runner.run_sync(triage, question)


Key difference: routing logic lives in the instructions string,
not in a Python routing function. The model decides at runtime.
```

### When to Use Each

| Scenario | Use Handoffs | Use LangGraph |
|----------|-------------|---------------|
| Routing is expressible in English | ✓ | |
| Routing requires complex boolean logic | | ✓ |
| Need typed state shared across nodes | | ✓ |
| OpenAI-only model stack | ✓ | |
| Mixed providers (Anthropic, Ollama) | | ✓ |
| Rapid prototyping | ✓ | |

In [ ]:
# ── 4-A: Minimal handoff — one dispatcher, one specialist ─────────────────────
# The dispatcher cannot answer directly — it must always hand off.

# handoff() returns a Handoff object that tells the Runner to switch the active agent
from agents import handoff

sdk_expert = Agent(
    name="SDKExpert",
    model=MODEL,
    instructions=(
        "You are an expert on the OpenAI Agents SDK. "
        "Use keyword_search to ground every answer in documented facts, "
        "then give a clear 2-3 sentence response."
    ),
    tools=[keyword_search],
)

dispatcher = Agent(
    name="Dispatcher",
    model=MODEL,
    instructions=(
        "You are a dispatcher. Do NOT answer questions directly. "
        "Always hand off every question to SDKExpert."
    ),
# the model calls handoff as a tool; the Runner intercepts it and routes to the target agent
    handoffs=[handoff(sdk_expert)],
)

result = Runner.run_sync(dispatcher, "How does Runner.run_sync() work?")
print(f"Final answer from: {result.last_agent.name}")
print(f"Answer: {result.final_output}")

In [ ]:
# ── 4-B: Verify that handoff actually transferred control ─────────────────────
# result.last_agent tells you which agent produced the final output.
# result.new_items shows every step including the handoff transfer.

result = Runner.run_sync(dispatcher, "Explain the @tool decorator.")

print(f"Started with: {dispatcher.name}")
print(f"Finished with: {result.last_agent.name}")
print(f"Total items in run: {len(result.new_items)}")
print()

for i, item in enumerate(result.new_items):
    print(f"  Step {i}: {type(item).__name__}")

print()
print(f"Final answer:\n{result.final_output}")

---

## Part 5 — Multi-Agent System: Triage → Researcher + Writer

---

### The Pattern

A **triage agent** sits at the entry point. It reads the user's request and hands off to the most appropriate specialist. Specialists never talk to each other — they receive the full conversation from triage and respond to the user.

```
  User question
       │
       ▼
  ┌──────────┐
  │  Triage  │  ← decides routing, never answers directly
  └────┬─────┘
       │
       ├── factual question ──────► ┌────────────┐
       │                            │ Researcher │ ← keyword_search(@tool) → answer
       │                            └────────────┘
       │
       └── explain / summarize ───► ┌────────┐
                                     │ Writer │ ← rewrites into polished prose
                                     └────────┘
```

This pattern maps directly to the `src/workflow.py` in this example's source files.

In [ ]:
# ── 5-A: Build the full triage system ────────────────────────────────────────

researcher_agent = Agent(
    name="Researcher",
    model=MODEL,
    instructions=(
        "You answer factual questions about the OpenAI Agents SDK. "
        "Always call keyword_search first to retrieve relevant facts, "
        "then compose a clear 2-3 sentence answer."
    ),
    tools=[keyword_search],
)

writer_agent = Agent(
    name="Writer",
    model=MODEL,
    instructions=(
        "You receive research findings or raw facts and rewrite them as a polished, "
        "readable paragraph suitable for a technical blog post. "
        "Aim for 3-5 sentences with a clear topic sentence."
    ),
)

triage_agent = Agent(
    name="Triage",
    model=MODEL,
    instructions=(
        "Route the user's request: "
        "if they ask a factual question (what, when, how does X work), hand off to Researcher. "
        "If they ask to explain, summarize, rewrite, or polish something, hand off to Writer. "
        "Do not answer directly — always hand off."
    ),
# listing both handoffs lets triage choose — the model picks based on instructions + question
    handoffs=[handoff(researcher_agent), handoff(writer_agent)],
)

print("System ready:")
print(f"  Triage     → handoffs: {[h.agent_name for h in triage_agent.handoffs]}")
print(f"  Researcher → tools:    {[t.name for t in researcher_agent.tools]}")
print(f"  Writer     → tools:    {[t.name for t in writer_agent.tools]}")

In [ ]:
# ── 5-B: Test routing — does Triage send the right question to the right agent?
# Factual questions → Researcher; writing requests → Writer.

test_cases = [
    ("What are the core primitives of the OpenAI Agents SDK?", "Researcher"),
    ("How does built-in tracing work in the Agents SDK?", "Researcher"),
    ("How does the Agents SDK differ from LangGraph?", "Researcher"),
    ("Rewrite this for a blog post: 'Agents SDK has tools and handoffs'", "Writer"),
    ("Explain the Agents SDK in simple terms for a non-technical audience", "Writer"),
]

print(f"{'Question':55} {'Expected':12} {'Got':12} {'Match'}")
print("-" * 95)

for question, expected in test_cases:
    result = Runner.run_sync(triage_agent, question)
    got = result.last_agent.name
    match = "PASS" if got == expected else "FAIL"
    print(f"{question[:54]:55} {expected:12} {got:12} {match}")

In [ ]:
# ── 5-C: Full output — see the complete answer with routing metadata ───────────

SAMPLE_QUESTIONS = [
    "What are the core primitives of the OpenAI Agents SDK?",
    "How does the Agents SDK differ from LangGraph?",
    "How does built-in tracing work in the Agents SDK?",
]

for q in SAMPLE_QUESTIONS:
    result = Runner.run_sync(triage_agent, q)
    print(f"Q: {q}")
    print(f"[Handled by: {result.last_agent.name}]")
    print(f"A: {result.final_output}")
    print()

---

## Part 6 — Tracing: What Happened Inside the Run?

---

### Built-in Tracing — Zero Config

One of the Agents SDK's biggest quality-of-life features: **every run is automatically traced**. You don't need to configure LangSmith, set up a callback, or instrument your code. The `RunResult.new_items` list contains every step that occurred.

### Comparing Tracing Approaches

| Approach | Setup required | Data location | Viewing UI |
|----------|---------------|---------------|------------|
| **Agents SDK (built-in)** | None — works out of the box | `RunResult.new_items` | OpenAI platform dashboard |
| **LangSmith** | `LANGCHAIN_API_KEY` + env vars | LangSmith cloud | LangSmith web UI |
| **Langfuse** | API keys + callback | Langfuse cloud or self-hosted | Langfuse web UI |
| **Custom callback** | Subclass `BaseCallbackHandler` | Wherever you write it | DIY |

### What `new_items` Contains

| Item type | What it represents |
|-----------|-------------------|
| `MessageOutputItem` | A text response from an agent |
| `ToolCallItem` | The model decided to call a tool |
| `ToolCallOutputItem` | The result returned by the tool |
| `HandoffOutputItem` | Control was transferred to another agent |

In [ ]:
# ── 6-A: Inspect a run with tool calls ───────────────────────────────────────
# Run the researcher (has keyword_search tool) and inspect every step.

trace_result = Runner.run_sync(
    researcher_agent,
    "What are the core primitives of the OpenAI Agents SDK?",
)

print("=== RUN TRACE ===")
print(f"Agent:        {trace_result.last_agent.name}")
print(f"Total items:  {len(trace_result.new_items)}")
print()

for i, item in enumerate(trace_result.new_items):
    item_type = type(item).__name__
    print(f"Step {i}: {item_type}")

    raw = item.raw_item if hasattr(item, "raw_item") else item
    if hasattr(raw, "content"):
        content = raw.content
        if isinstance(content, list):
            for part in content:
                if hasattr(part, "text") and part.text:
                    print(f"  text: {part.text[:120]}")
        elif isinstance(content, str):
            print(f"  content: {content[:120]}")
    print()

print(f"Final output: {trace_result.final_output}")

In [ ]:
# ── 6-B: Inspect a multi-agent run with a handoff ────────────────────────────
# Run the full triage system and trace all steps including the handoff transfer.

handoff_result = Runner.run_sync(
    triage_agent,
    "What are the core primitives of the OpenAI Agents SDK?",
)

print("=== MULTI-AGENT TRACE ===")
print(f"Started at:  Triage")
print(f"Ended at:    {handoff_result.last_agent.name}")
print(f"Steps:       {len(handoff_result.new_items)}")
print()

for i, item in enumerate(handoff_result.new_items):
    print(f"  [{i}] {type(item).__name__}")

print()
print(f"Answer: {handoff_result.final_output[:300]}")

---

## Part 7 — Guardrails: Validating Inputs Before Routing

---

### What Are Guardrails?

**Guardrails** are validation steps that run *before* your agent processes a request. They protect against:
- Off-topic questions (the user asks your SDK bot for cooking recipes)
- Prompt injection attempts
- Malformed or unexpectedly short/long inputs

The Agents SDK supports native `InputGuardrail` and `OutputGuardrail` objects. A clean lightweight alternative is a dedicated **gatekeeper agent** that runs first and either passes the request on or returns an error message.

### Guardrail Patterns

| Pattern | Mechanism | When to use |
|---------|-----------|------------|
| **Gatekeeper agent** | Run a cheap agent first; block if off-topic | Simple topic filtering |
| **`InputGuardrail`** | SDK-native; runs in parallel with agent | SDK >= 0.4, concurrent check |
| **Structured output + validation** | Agent returns JSON, validate with dataclass | Output shape enforcement |
| **Pre-flight `@tool`** | First tool call validates the question | When guardrail logic needs retrieval |

In [ ]:
# ── 7-A: Gatekeeper agent pattern ─────────────────────────────────────────────
# A cheap agent validates the input BEFORE handing off to the triage system.

gatekeeper = Agent(
    name="Gatekeeper",
    model=MODEL,
    instructions=(
        "You are a topic filter. "
        "Respond with exactly 'PASS' if the question is about the OpenAI Agents SDK, "
        "agent frameworks, LangGraph, AutoGen, or AI agents in general. "
        "Respond with exactly 'BLOCK: <reason>' if the question is off-topic. "
        "Do not answer the question — only classify it."
    ),
)


# two-pass pattern: cheap classification first cuts cost and latency for off-topic requests
def run_with_guardrail(question: str) -> str:
    """Run the gatekeeper first; only proceed to triage if it returns PASS."""
    gate_result = Runner.run_sync(gatekeeper, question)
    gate_response = gate_result.final_output.strip()

    if gate_response.startswith("PASS"):
        main_result = Runner.run_sync(triage_agent, question)
        return f"[{main_result.last_agent.name}] {main_result.final_output}"
    else:
        return f"[BLOCKED] {gate_response}"


test_inputs = [
    "What are the core primitives of the OpenAI Agents SDK?",  # on-topic
    "How does LangGraph compare to the Agents SDK?",           # on-topic
    "What is the best pasta recipe?",                          # off-topic
    "What is my bank account password?",                       # off-topic
]

for q in test_inputs:
    answer = run_with_guardrail(q)
    print(f"Q: {q}")
    print(f"A: {answer[:200]}")
    print()

In [ ]:
# ── 7-B: Structured output — enforce response shape with a dataclass ───────────
# Ask the model to return JSON; parse and validate it in Python.
# This is useful when downstream code depends on specific fields.

import json
from dataclasses import dataclass


@dataclass
class ClassifiedQuestion:
    topic: str        # one of: 'factual', 'writing', 'off-topic'
    confidence: str   # 'high', 'medium', 'low'
    reason: str       # one sentence explaining the classification


# Alternative: use output_type= on the Agent for native structured output without manual json.loads()
classifier_agent = Agent(
    name="Classifier",
    model=MODEL,
    instructions=(
        "Classify the user's question and respond with a JSON object with three fields: "
        "'topic' (one of: factual, writing, off-topic), "
        "'confidence' (one of: high, medium, low), "
        "'reason' (one sentence). "
        "Respond with valid JSON only, no markdown fences."
    ),
)

questions = [
    "What are the core primitives of the OpenAI Agents SDK?",
    "Write a blog post intro about AI agents",
    "What is the weather in Paris?",
]

for q in questions:
    result = Runner.run_sync(classifier_agent, q)
# parsing inline lets us validate the shape; a malformed response triggers the except branch
    try:
        raw = result.final_output.strip()
        data = json.loads(raw)
        classified = ClassifiedQuestion(**data)
        print(f"Q: {q}")
        print(f"   topic={classified.topic}  confidence={classified.confidence}")
        print(f"   reason: {classified.reason}")
    except (json.JSONDecodeError, TypeError) as e:
        print(f"Q: {q}")
        print(f"   Parse error: {e} — raw: {result.final_output[:80]}")
    print()

---

## Exercises

---

### Exercise 1 — Add a Third Specialist

Create a `CodeAgent` that generates Python code snippets demonstrating Agents SDK concepts. Update `Triage`'s instructions to route code requests to it.

**Test with:** `"Show me Python code to create a simple Agent"` and `"Give me an example of a @tool function"`

---

### Exercise 2 — Shared Tool with Writer

Give `keyword_search` to `writer_agent` as well. Ask a writing question like `"Write a blog paragraph about handoffs"`. Does the Writer call `keyword_search` before writing? What does this tell you about how tools affect agent behavior?

---

### Exercise 3 — LangGraph Comparison

Implement the same triage flow (factual → Researcher, writing → Writer) using LangGraph with `add_conditional_edges`. Compare:
- Lines of code
- How routing logic is expressed (Python function vs. natural language)
- How you would add a third specialist to each version

---

### Exercise 4 — Deep Trace Inspection

Run a question through the full triage system and print every `new_items` entry with its type and any text content. Find the exact step where the handoff occurs. How many steps happen before the final answer?

In [ ]:
# ── Exercise 1 Starter — add a CodeAgent to the triage system ─────────────────

# TODO: create a CodeAgent that generates Python code snippets
code_agent = Agent(
    name="CodeAgent",
    model=MODEL,
    instructions=(
        # TODO: write instructions that make this agent generate clean Python code
        "..."
    ),
)

# TODO: create an updated triage agent that includes code_agent in its handoffs
# and updates the instructions to route code requests to CodeAgent
triage_v2 = Agent(
    name="TriageV2",
    model=MODEL,
    instructions=(
        # TODO: extend the routing instructions to cover code requests
        "..."
    ),
    handoffs=[
        # TODO: add handoffs to researcher_agent, writer_agent, and code_agent
    ],
)

# TODO: test with these questions — verify the right specialist handles each
code_questions = [
    "Show me Python code to create a simple Agent using the Agents SDK",
    "Give me an example of a @tool function",
    "What are the core primitives of the OpenAI Agents SDK?",  # should still go to Researcher
]

for q in code_questions:
    # TODO: run triage_v2 and print the agent name and answer
    pass

In [ ]:
# ── Exercise 4 Starter — deep trace inspection ────────────────────────────────

question_to_trace = "What are the core primitives of the OpenAI Agents SDK?"

# TODO: run triage_agent with the question above
# trace_r = Runner.run_sync(triage_agent, question_to_trace)

# TODO: loop over trace_r.new_items
# Print: step number, item type, and any text content you can extract
# Identify: at which step does the handoff happen?
pass

---

## Answer Key

Use this section after attempting the exercises yourself. These are sample solutions — not the only correct approach.

---

### Exercise 1 — Add a Third Specialist

**What good output looks like:** The `CodeAgent` receives code requests and returns syntactically valid Python using only `agents` SDK imports. Triage correctly routes `"Show me Python code..."` to `CodeAgent` and `"What are the core primitives..."` to `Researcher` as before.

**Key insight:** The routing instructions must be specific enough that the model distinguishes "tell me about X" (factual → Researcher) from "show me code for X" (code → CodeAgent). If routing fails, add more examples to the instructions.

### Exercise 2 — Shared Tool with Writer

**Expected behavior:** When `Writer` has `keyword_search`, it may call it before writing — using the search results as raw material, then transforming them into polished prose. The model infers from the docstring *when* the tool is appropriate.

**Key insight:** Giving a tool to an agent changes its behavior even if the instructions don't explicitly say to use it.

### Exercise 3 — LangGraph Comparison

**Key differences:** LangGraph requires roughly twice as much code because you must define a `TypedDict` state, write a routing function, and wire `add_conditional_edges`. Adding a third specialist in LangGraph requires updating the graph and the routing function. In Agents SDK, you only update `handoffs` and `instructions`. For simple chat routing, Agents SDK is significantly leaner.

### Exercise 4 — Deep Trace Inspection

**Expected structure for a triage → researcher run with a tool call:**
```
Step 0: MessageOutputItem       ← Triage decides to hand off
Step 1: HandoffOutputItem       ← handoff transfer recorded
Step 2: ToolCallItem            ← Researcher calls keyword_search
Step 3: ToolCallOutputItem      ← keyword_search returns results
Step 4: MessageOutputItem       ← Researcher's final answer
```
The handoff occurs at Step 1. The tool call loop (Steps 2-3) may repeat if the model calls the tool multiple times before answering.

In [ ]:
# Answer Key — Exercise 1: CodeAgent

code_agent_answer = Agent(
    name="CodeAgent",
    model=MODEL,
    instructions=(
        "You generate clean, runnable Python code examples that demonstrate "
        "OpenAI Agents SDK concepts. Always include imports. "
        "Keep examples under 20 lines. Add a one-line comment explaining each key step."
    ),
)

triage_v2_answer = Agent(
    name="TriageV2",
    model=MODEL,
    instructions=(
        "Route the user's request to the best specialist: "
        "- Factual questions (what, when, how does X work) → Researcher. "
        "- Code requests (show me, give me an example, write code) → CodeAgent. "
        "- Explain, summarize, rewrite, polish → Writer. "
        "Do not answer directly — always hand off."
    ),
    handoffs=[
        handoff(researcher_agent),
        handoff(writer_agent),
        handoff(code_agent_answer),
    ],
)

code_questions = [
    "Show me Python code to create a simple Agent using the Agents SDK",
    "Give me an example of a @tool function",
    "What are the core primitives of the OpenAI Agents SDK?",
]

for q in code_questions:
    r = Runner.run_sync(triage_v2_answer, q)
    print(f"Q: {q}")
    print(f"[Handled by: {r.last_agent.name}]")
    print(f"A: {r.final_output[:300]}")
    print()

In [ ]:
# Answer Key — Exercise 4: Deep trace inspection

question_to_trace = "What are the core primitives of the OpenAI Agents SDK?"
trace_r = Runner.run_sync(triage_agent, question_to_trace)

print(f"Question: {question_to_trace}")
print(f"Final agent: {trace_r.last_agent.name}")
print(f"Total steps: {len(trace_r.new_items)}")
print()
print("=== FULL TRACE ===")

for i, item in enumerate(trace_r.new_items):
    item_type = type(item).__name__
    print(f"Step {i}: {item_type}")

    raw = item.raw_item if hasattr(item, "raw_item") else None
    if raw is not None:
        if hasattr(raw, "content") and isinstance(raw.content, list):
            for part in raw.content:
                if hasattr(part, "text") and part.text:
                    print(f"  → {part.text[:120]}")
        elif hasattr(raw, "output"):
            print(f"  → output: {str(raw.output)[:120]}")
    print()

print(f"Final answer:\n{trace_r.final_output}")

---

## What's Next?

You now have a working multi-agent system with triage, tools, handoffs, tracing, and guardrails. Here's where to go deeper:

### Explore adjacent patterns in this repo
- **Example 31 — Multi-Agent Debate** ([`../31-multi-agent-debate/`](../31-multi-agent-debate/)): agents that argue against each other before a judge decides — a different kind of multi-agent coordination.
- **Example 6 — Multi-Agent Graph** ([`../6-multi-agent-graph/`](../6-multi-agent-graph/)): the same triage pattern implemented with LangGraph `add_conditional_edges` — compare the code directly.
- **Example 11 — HITL Approval** ([`../11-hitl-approval/`](../11-hitl-approval/)): add a human-in-the-loop gate between triage and specialist.
- **Example 61 — Guardrails Agent** ([`../61-guardrails-agent/`](../61-guardrails-agent/)): full guardrails pipeline with Pydantic validation at the LangGraph level.

### Strengthen the triage system you built
- **Streaming:** Try `Runner.run_streamed()` (async) to stream the Researcher's answer token-by-token
- **Parallel tool calls:** Give an agent two `@tool` functions and ask a question that requires both in one response
- **Conversation memory:** Pass conversation history back into `Runner.run_sync()` to maintain context across multiple turns

### Go to production
- **Async execution:** Swap `Runner.run_sync()` → `await Runner.run()` for use in FastAPI/Starlette servers
- **OpenAI tracing dashboard:** Log in to https://platform.openai.com and view your agent traces in the Traces tab — every run from this notebook is already there
- **Native guardrails API:** Explore `InputGuardrail` and `OutputGuardrail` in the SDK for concurrent validation without a separate gatekeeper agent

### Further reading
- OpenAI Agents SDK docs: https://openai.github.io/openai-agents-python/
- OpenAI function calling guide: https://platform.openai.com/docs/guides/function-calling
- Yao et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* https://arxiv.org/abs/2210.03629
- Xi et al. (2023). *The Rise and Potential of LLM-Based Agents: A Survey.* https://arxiv.org/abs/2309.07864

---

*Workshop complete. The logical next step is example 6 — see how the same multi-agent triage pattern looks when expressed as an explicit LangGraph state machine.*